In [1]:
# Gather cojo_output files, extract gene names, parse each gene output

In [15]:
import sys
import os
import re
import pandas as pd
import parsl
import subprocess
from glob import glob
from subprocess import call
from parsl.app.app import bash_app, python_app, join_app
from parsl.dataflow.futures import AppFuture
from parsl.data_provider.files import File

In [20]:
def parse_gene_gwas(gene_file, gene_name, study, inputs=[], outputs=[],
                      stdout=parsl.AUTO_LOGNAME, 
                      stderr=parsl.AUTO_LOGNAME): 
    import os
    import numpy as np
    import pandas as pd
    if not os.path.exists(os.path.dirname(outputs[0])):
        os.makedirs(os.path.dirname(outputs[0]), exist_ok=True)
    if os.path.exists(outputs[0]):
        return("echo 'Output exists. Remove it or delete it.'")

    output = outputs[0]
    bash_command = \
    f"""
    source activate /media/desk15/iy2120/miniconda3/envs/TWAS
    
    python3 /media/desk15/iy2120/TWAS/Breast-cancer-Example/summary-gwas-imputation/src/gwas_parsing.py \
    -gwas_file {gene_file} \
    -output_column_map SNP variant_id \
    -output_column_map refA effect_allele \
    -output_column_map non_effect_allele non_effect_allele \
    -output_column_map bC effect_size \
    -output_column_map bC_se standard_error \
    -output_column_map Chr chromosome --chromosome_format -output_column_map bp position \
    -output_column_map pC pvalue \
    -output_order variant_id panel_variant_id chromosome position effect_allele non_effect_allele frequency pvalue zscore effect_size standard_error sample_size n_cases \
    -snp_reference_metadata /media/desk15/iy2120/TWAS/Breast-cancer-Example/data/reference/gtex_v8_eur_filtered_maf0.01_monoallelic_variants.txt.gz METADATA  \
    -output {output}
    """
    
    proc = subprocess.run(bash_command, shell=True, executable="/bin/bash")
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed: {bash_command}")
    return

In [9]:
study = "final_metal_ad_kunkle_pgcalz_ukb"

In [12]:
cojo_out_files = glob(f"/media/desk15/iy2120/Project2026/Myproject_AD/05_cojo_ctwas/output/intermediate_data/cojo_output/{study}/*.cma.cojo")

In [13]:
cojo_out_files

['/media/desk15/iy2120/Project2026/Myproject_AD/05_cojo_ctwas/output/intermediate_data/cojo_output/final_metal_ad_kunkle_pgcalz_ukb/ENSG00000198502.5.cma.cojo',
 '/media/desk15/iy2120/Project2026/Myproject_AD/05_cojo_ctwas/output/intermediate_data/cojo_output/final_metal_ad_kunkle_pgcalz_ukb/ENSG00000177045.7.cma.cojo',
 '/media/desk15/iy2120/Project2026/Myproject_AD/05_cojo_ctwas/output/intermediate_data/cojo_output/final_metal_ad_kunkle_pgcalz_ukb/ENSG00000197093.10.cma.cojo',
 '/media/desk15/iy2120/Project2026/Myproject_AD/05_cojo_ctwas/output/intermediate_data/cojo_output/final_metal_ad_kunkle_pgcalz_ukb/ENSG00000189114.6.cma.cojo',
 '/media/desk15/iy2120/Project2026/Myproject_AD/05_cojo_ctwas/output/intermediate_data/cojo_output/final_metal_ad_kunkle_pgcalz_ukb/ENSG00000130204.12.cma.cojo',
 '/media/desk15/iy2120/Project2026/Myproject_AD/05_cojo_ctwas/output/intermediate_data/cojo_output/final_metal_ad_kunkle_pgcalz_ukb/ENSG00000168090.9.cma.cojo',
 '/media/desk15/iy2120/Project20

In [16]:
cojo_out_gene_names = [re.match("(ENSG\d{1,}\.\d{1,})\.cma\.cojo", thing)[1] for thing in map(os.path.basename, cojo_out_files)]

In [17]:
cojo_out_gene_names

['ENSG00000198502.5',
 'ENSG00000177045.7',
 'ENSG00000197093.10',
 'ENSG00000189114.6',
 'ENSG00000130204.12',
 'ENSG00000168090.9',
 'ENSG00000179344.16',
 'ENSG00000196666.4',
 'ENSG00000011478.11',
 'ENSG00000108395.13',
 'ENSG00000188186.10',
 'ENSG00000142233.11',
 'ENSG00000104856.13',
 'ENSG00000245975.2',
 'ENSG00000134571.10',
 'ENSG00000030582.17',
 'ENSG00000166529.14',
 'ENSG00000214787.9',
 'ENSG00000185294.6',
 'ENSG00000012061.15',
 'ENSG00000140090.17',
 'ENSG00000106261.16',
 'ENSG00000166526.16',
 'ENSG00000109919.9',
 'ENSG00000125755.18',
 'ENSG00000134569.9',
 'ENSG00000134575.9',
 'ENSG00000143479.15',
 'ENSG00000271046.1',
 'ENSG00000186567.12',
 'ENSG00000125746.16',
 'ENSG00000170608.2',
 'ENSG00000104859.14',
 'ENSG00000239521.7',
 'ENSG00000170921.15',
 'ENSG00000221838.9',
 'ENSG00000213619.9',
 'ENSG00000028528.14',
 'ENSG00000157734.13',
 'ENSG00000105383.14',
 'ENSG00000104941.7',
 'ENSG00000216588.8',
 'ENSG00000231852.6',
 'ENSG00000010310.8',
 'ENSG00

In [21]:
gene_parsed_res = []
for gene_file, gene_name in zip(cojo_out_files, cojo_out_gene_names):
	out_file = File(f"/media/desk15/iy2120/Project2026/Myproject_AD/05_cojo_ctwas/output/intermediate_data/condTWAS_parse/{study}/{gene_name}.txt.gz")
	gene_parsed_res.append(parse_gene_gwas(gene_file, gene_name, study=study, inputs=None, outputs=[out_file]))

INFO - Parsing input GWAS
INFO - loaded 55 variants
INFO - Creating index to attach reference ids
INFO - Acquiring reference metadata
INFO - alligning alleles
INFO - 55 variants after restricting to reference variants
INFO - Ensuring variant uniqueness
INFO - 55 variants after ensuring uniqueness
INFO - Checking for missing frequency entries
INFO - Saving...
INFO - Finished converting GWAS in 42.23406860604882 seconds
INFO - Parsing input GWAS
INFO - loaded 101 variants
INFO - Creating index to attach reference ids
INFO - Acquiring reference metadata
INFO - alligning alleles
INFO - 96 variants after restricting to reference variants
INFO - Ensuring variant uniqueness
INFO - 96 variants after ensuring uniqueness
INFO - Checking for missing frequency entries
INFO - Saving...
INFO - Finished converting GWAS in 41.44664812646806 seconds
INFO - Parsing input GWAS
INFO - loaded 17 variants
INFO - Creating index to attach reference ids
INFO - Acquiring reference metadata
INFO - alligning alle